## Imports & Connection

In [1]:
from pyhive import presto
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np

from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression

import seaborn as sns

conn = presto.connect(
        host="presto-gateway.serving.data.production.internal",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )

import warnings
warnings.filterwarnings("ignore")



## Filters

In [2]:
city = 'Chennai'
#service='Auto'
start_date = '20240727'
end_date = '20241027'
end_date2 = '2024-10-27'

## Data Query 

In [3]:
query = f"""
with fe_tbl as (
        
        SELECT
            current_city AS city,
            user_id as customer_id,
            fare_estimate_id,
            service_details_id,
            service_name,
            offer_type,
            discount_type,
            quarter_hour,
            cast(discount_amount AS double) AS discount,
            cast(final_amount AS double) AS finalamount,
            cast(hf_distance AS double) AS distance,
            cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone || ' - ' || service_name AS unique_id,
            date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
            cast(sub_total AS double) AS subtotal,
            row_number() over (partition by cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone || ' - ' || service_name order by epoch desc) as updated_seq
        FROM
            canonical.clevertap_customer_fare_estimate
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND current_city = '{city}'
            AND service_name in ('Auto','Link')
    ),
    
    rr_tbl as (
        
        SELECT
            current_city AS city,
            user_id as customer_id,
            fare_estimate_id,
            service_name,
            cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone || ' - ' || service_name AS unique_id,
            date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
            cast(sub_total AS double) AS subtotal,
            row_number() over (partition by cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone || ' - ' || service_name order by epoch desc) as updated_seq
        FROM
            canonical.clevertap_customer_request_rapido
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND current_city = '{city}'
            AND service_name in ('Auto','Link')
    ),
    
    response_tbl AS (
    
        SELECT
            fare_estimate_id,
            service_level,
            CAST(dynamic_surge_amount AS DOUBLE) AS dynamic_surge,
            CAST(dynamic_fare_amount AS DOUBLE) AS dynamic_fare,
            computed_pickup_cluster AS pickup_cluster,
            computed_drop_cluster AS drop_cluster,
            surge_strategy,
            CAST(rate_card_amount AS DOUBLE) AS rate_card
        FROM
            experiments.iprice_cleaned_responses_v2
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city = '{city}'
            AND service_level in ('Auto','Link')
    ),
    
    rr_merged AS (
        
        SELECT
            city,
            customer_id,
            service_name,
            rr_tbl.fare_estimate_id AS fare_estimate_id,
            unique_id,
            pickup_cluster,
            drop_cluster,
            orderdate,
            subtotal,
            dynamic_surge,
            dynamic_fare,
            surge_strategy,
            1.0 AS rr_count
        FROM
            rr_tbl
        LEFT JOIN
            response_tbl
        ON rr_tbl.fare_estimate_id = response_tbl.fare_estimate_id
        and rr_tbl.service_name = response_tbl.service_level
        WHERE updated_seq = 1
    ),
    
    fe_merged AS (
        
        SELECT
            city,
            customer_id,
            service_name,
            fe_tbl.fare_estimate_id AS fare_estimate_id,
            unique_id,
            service_details_id,
            pickup_cluster,
            drop_cluster,
            orderdate,
            subtotal,
            dynamic_surge,
            dynamic_fare,
            surge_strategy,
            finalamount,
            distance,
            offer_type,
            rate_card,
            discount_type,
            discount,
            quarter_hour,
            1.0 AS fe_count
        FROM
            fe_tbl
        LEFT JOIN
            response_tbl
        ON fe_tbl.fare_estimate_id = response_tbl.fare_estimate_id
        and fe_tbl.service_name = response_tbl.service_level
        WHERE updated_seq = 1
    
    ),
    
    fe_rr AS (
    
        SELECT
            
            fe_merged.city AS city,
            fe_merged.customer_id AS customer_id,
            fe_merged.fare_estimate_id AS fare_estimate_id,
            fe_merged.unique_id AS unique_id,
            fe_merged.pickup_cluster AS pickup_cluster,
            fe_merged.drop_cluster AS drop_cluster,
            fe_merged.service_details_id,
            fe_merged.service_name,
            fe_merged.orderdate AS orderdate,
            fe_merged.distance,
            fe_merged.offer_type,
            fe_merged.rate_card,
            fe_merged.discount_type,
            fe_merged.discount,
            fe_merged.finalamount,
            fe_merged.quarter_hour,
            CASE 
                WHEN (fe_merged.surge_strategy = 'mismatch_generic' OR fe_merged.surge_strategy = 'mismatch_gradient') AND (fe_merged.dynamic_surge+fe_merged.dynamic_fare)>0 THEN 'Mismatch_Surge' 
                WHEN (fe_merged.surge_strategy != 'mismatch_generic' AND fe_merged.surge_strategy != 'mismatch_gradient') AND (fe_merged.dynamic_surge+fe_merged.dynamic_fare)>0 THEN 'Non-Mismatch_Surge'
                ELSE 'Non-Surged' END AS surge_type,
            fe_merged.subtotal AS subtotal,
            fe_merged.dynamic_surge AS dynamic_surge,
            fe_merged.dynamic_fare AS dynamic_fare,
            (fe_merged.dynamic_surge + fe_merged.dynamic_fare) AS total_surge,
            fe_merged.fe_count AS fe_count,
            coalesce(rr_merged.rr_count,0) AS rr_count
        FROM
            fe_merged
        LEFT JOIN
            rr_merged
        ON fe_merged.city = rr_merged.city
        AND fe_merged.unique_id = rr_merged.unique_id
        AND fe_merged.orderdate = rr_merged.orderdate
    ),
    
    data as
(
        select
        run_date as run_day,
        run_date,
        customer_id as customerId,
        taxi_lifetime_last_ride_city as city_name
        from
        datasets.iallocator_customer_segments
        where
        run_date = '{end_date2}'
        and taxi_lifetime_last_ride_city = '{city}'
),

    persona0 as
(
        select
        a.*,
        b.*,
        (finalamount - rate_card) * 100.0 / rate_card as pct_delta_from_rc
        from 
            data a 
        join 
            fe_rr b 
        on a.customerId = b.customer_id
),

persona as
(
select
*,
case
    when pct_delta_from_rc < -28 then '0. P<-28'
    when (pct_delta_from_rc >= -28 and pct_delta_from_rc < -21) then '1.-28<=P<-21'
    when (pct_delta_from_rc >= -21 and pct_delta_from_rc < -14) then '2.-21<=P<-14'
    when (pct_delta_from_rc >= -14 and pct_delta_from_rc < -7) then '3.-14<=P<-7'
    when (pct_delta_from_rc >= -7 and pct_delta_from_rc < 0) then '4.-7<=P<0'
    when (pct_delta_from_rc >= 0 and pct_delta_from_rc < 7) then '5.0<=P<7'
    when (pct_delta_from_rc >= 7 and pct_delta_from_rc < 14) then '6.7<=P<14'
    when (pct_delta_from_rc >= 14 and pct_delta_from_rc < 21) then '7.14<=P<21'
    when (pct_delta_from_rc >= 21 and pct_delta_from_rc < 28) then '8.21<=P<28'
    else '9. 28<=P' end new_price_point


    from persona0
),

agg as
(
select
city_name,customer_id,service_name,
count(distinct new_price_point) new_price_point,
sum(fe_count) fe_count,
sum(rr_count) rr_count,
sum(rr_count)*100.00/sum(fe_count) as fe2rr
from persona
group by 1,2,3
having
sum(fe_count) >=4
and sum(rr_count) >= 2
and count(distinct new_price_point) >=3

)

select
city_name,customer_id,service_name,
new_price_point as pp,
sum(fe_count) fe_count,
sum(rr_count) rr_count,
sum(rr_count)*100.00/sum(fe_count) as fe2rr
from persona 
group by 1,2,3,4



"""

In [4]:
data = pd.read_sql(query,conn)

DatabaseError: {'message': 'Query exceeded the maximum execution time limit of 10.00m', 'errorCode': 131075, 'errorName': 'EXCEEDED_TIME_LIMIT', 'errorType': 'INSUFFICIENT_RESOURCES', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'Query exceeded the maximum execution time limit of 10.00m', 'suppressed': [], 'stack': ['io.trino.execution.QueryTracker.enforceTimeLimits(QueryTracker.java:187)', 'io.trino.execution.QueryTracker.lambda$start$0(QueryTracker.java:89)', 'com.google.common.util.concurrent.MoreExecutors$ScheduledListeningDecorator$NeverSuccessfulListenableFutureTask.run(MoreExecutors.java:730)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)', 'java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)', 'java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.jav

In [ ]:
data.head(5)

In [ ]:
data.shape

## Process

In [ ]:
# Converting pp column (9. 28<=P) to (9)
data['pp'] = data['pp'].apply(lambda x: int(x[0]))

In [ ]:
data1 = data\
            .groupby(['city_name','customer_id','service_name'])\
            .agg({'fe_count':'sum','rr_count':'sum','pp':'count'})\
            .reset_index()

In [ ]:
data1[data1['customer_id']=='63703a3fb48cdd5e1af99a2b']

In [ ]:
data1[data1['pp'] >= 3 ].head(5)

### Initialize

In [ ]:
# Set Plot Figure Size:
plt.rcParams['figure.figsize'] = [16, 9]

# Initialize an Empty DataFrame
coef_df=pd.DataFrame()

In [ ]:
data\
.groupby(['customer_id','service_name','pp'])\
.agg({'fe_count':'sum','rr_count':'sum'})\
.reset_index().head(5)

In [ ]:
temp=data.groupby(['customer_id','service_name','pp']).agg({'fe_count':'sum','rr_count':'sum'}).reset_index()
temp['fe2rr']=temp['rr_count']/temp['fe_count']

In [ ]:
temp.head(5)

In [ ]:
# Loop Through Unique Customer and Service Combinations

for item in data['customer_id'].unique()[1:10000]: # 
    
    for jtem in data['service_name'].unique():
        
        try:
            # Filter Data and Apply Linear Regression
            cust_data = temp[(temp['customer_id']==item) & (temp['service_name']==jtem)]
            x1=np.array(np.log(1+cust_data[['pp']]))
            y=np.array(cust_data['fe2rr'])

            
            # Fit the Linear Regression Model
            reg = LinearRegression()
            reg_log1=reg.fit(x1,y)
            d = {'customer_id': item,'service_name': jtem, 'intercept':reg_log1.intercept_,'coef': reg_log1.coef_}
            df = pd.DataFrame(data=d)
            coef_df=coef_df.append(df)

        except:
            pass

In [ ]:
coef_df.head(5)

In [ ]:
coef_df_final=pd.merge(coef_df,
                       data1,
                       on=['customer_id','service_name'],
                       how='left')

coef_df_final['confidence_on_ps_signal']=coef_df_final.apply(lambda x: 'High' 
                                                             if x['fe_count']>=6 and x['rr_count']>=3 and x['pp']>=4 
                                                             else 'Low',
                                                             axis=1)
coef_df_final['run_date']=end_date2

In [ ]:
def ps_tag_2(x,city,service):
    
    coeff_40 = coef_df_final[(coef_df_final['service_name']==service) & (coef_df_final['city_name']==city)]['coef'].describe(percentiles=[0.40])['40%']
    coeff_60 = coef_df_final[(coef_df_final['service_name']==service) & (coef_df_final['city_name']==city)]['coef'].describe(percentiles=[0.60])['60%']

    if x <= coeff_40:
        return 'PS'
    elif x > coeff_40 and x <= coeff_60:
        return 'Grey'
    else: 
        return 'NPS'

In [ ]:
coef_df_final['PS_tag']= coef_df_final.apply(lambda x: ps_tag_2(x['coef'],x['city_name'],x['service_name']), axis=1)

In [ ]:
coef_df_final

In [ ]:
coef_df_final.head()

### Rest

In [12]:
# I just took 10000 customers for sample
base_query = f''' select distinct _id as customer_id from entity.users_snapshot ''' 

In [13]:
base = pd.read_sql(base_query,conn) # This is just sample base of 10000 customers

In [22]:
final_coef_df = pd.merge(base,coef_df_final, on = 'customer_id', how = 'outer')

In [24]:
final_coef_df.fillna('unknown',inplace=True)

In [26]:
final_coef_df

,customer_id,service_name,intercept,coef,city_name,fe_count,rr_count,pp,confidence_on_ps_signal,run_date,PS_tag
0,5de8c290782c906b1dec7c83,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,5de8c2c3782c906b1dec7cd6,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,5de8c43423da8b6b4b80536c,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,5de8c456782c906b1dec7f2c,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,5de8c47427a8c16b23f542e3,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
...,...,...,...,...,...,...,...,...,...,...,...
10012,5cdc896825ee3218d4c7ab67,Link,0.963503,-0.054671,Patna,9.0,7.0,3.0,Low,2022-11-06,PS
10013,5cdc896825ee3218d4c7ab67,Auto,0.0,0.0,Patna,9.0,0.0,3.0,Low,2022-11-06,PS
10014,5d042c72abbf484763822c33,Auto,0.0,0.0,Patna,5.0,0.0,4.0,Low,2022-11-06,PS
10015,5d1bb5c83b752c45cf93f3ad,Link,0.0,0.0,Patna,1.0,0.0,1.0,Low,2022-11-06,Grey


In [25]:
final_coef_df.to_csv('sample_data_with_all_base.csv',index= False) # Sample data with 100017 customers